# 팀 통합 다섯 조건 비교 실험
기본 모델2개와 파인튜닝3개의 비교 구성임. 별도 코드 ZIP 없이 실행 가능함. 데이터 검수 및 분할 검사를 통과한 뒤 RUN_TRAINING=True 설정 필요함. Test는 별도 확정 후 실행함. 실제 팀 학습은 미실행 상태임.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE = Path('/content/drive/MyDrive/ICT_통합실험0929')
for folder in ('inputs', 'weights', 'runs', 'results'):
    (BASE / folder).mkdir(parents=True, exist_ok=True)
print('작업 폴더:', BASE)


In [ ]:
import json, sys, os, subprocess, zipfile
CODE = Path('/content/crane_team0929')
CODE.mkdir(exist_ok=True)
CODE_FILES = {'metrics.py': '"""Shared binary pixel metrics. Undefined quantities stay null, never invented as 1."""\nimport numpy as np\n\n\ndef metrics(pred, gt, valid):\n    pred, gt, valid = [np.asarray(a, bool) for a in (pred, gt, valid)]\n    if pred.shape != gt.shape or gt.shape != valid.shape or not valid.any():\n        raise ValueError(\'Metric arrays must have equal shape and valid pixels\')\n    tp = int((pred & gt & valid).sum())\n    fp = int((pred & ~gt & valid).sum())\n    fn = int((~pred & gt & valid).sum())\n    tn = int((~pred & ~gt & valid).sum())\n    return dict(tp=tp, fp=fp, fn=fn, tn=tn, target_pixels=tp+fn, valid_pixels=int(valid.sum()),\n                iou=tp/(tp+fp+fn) if tp+fp+fn else None,\n                dice=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else None,\n                recall=tp/(tp+fn) if tp+fn else None,\n                precision=tp/(tp+fp) if tp+fp else (0. if tp+fn else None))\n\n\ndef summarize(rows, normal_fp_pixels=0):\n    """Image FP iff pixel count > configured threshold; default is any predicted pixel."""\n    def average(rs, key):\n        values = [r[key] for r in rs if r[key] is not None]\n        return float(np.mean(values)) if values else None\n    positive = [r for r in rows if r[\'target_pixels\'] > 0]\n    normal = [r for r in rows if r[\'normal_confirmed\']]\n    count = sum(r[\'fp\'] > normal_fp_pixels for r in normal)\n    tp, fp, fn = [sum(r[k] for r in rows) for k in (\'tp\', \'fp\', \'fn\')]\n    return dict(n_images=len(rows), positive_n=len(positive), normal_n=len(normal),\n                positive_macro={k: average(positive, k) for k in (\'iou\', \'dice\', \'precision\', \'recall\')},\n                micro_iou=tp/(tp+fp+fn) if tp+fp+fn else None,\n                normal_false_positive_images=count,\n                normal_false_positive_image_rate=count/len(normal) if normal else None,\n                normal_mean_false_positive_area=float(np.mean([r[\'fp\']/r[\'valid_pixels\'] for r in normal])) if normal else None,\n                normal_fp_pixels_threshold=normal_fp_pixels)\n\n\ndef grouped_summary(rows, normal_fp_pixels=0):\n    # Intentionally no pooled real+synthetic headline: the two test populations differ.\n    return {kind: summarize([r for r in rows if r[\'data_type\'] == kind], normal_fp_pixels)\n            for kind in (\'real\', \'synthetic\')}\n', 'run.py': '"""Train on train/val only; explicitly lock a run before one-time final Test."""\nimport os\nos.environ.setdefault(\'CUBLAS_WORKSPACE_CONFIG\', \':4096:8\')\nimport argparse\nimport csv\nimport json\nimport platform\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, RandomSampler\nimport segmentation_models_pytorch as smp\nfrom data import sha, save_json, verify_prepared, check_leakage\nfrom engine import Samples, checkpoint_state, evaluate, loss_value, MEAN, STD\n\nDEFAULTS = dict(size=512, epochs=20, patience=5, batch_size=2, lr=3e-5,\n                threshold=0.5, normal_fp_pixels=0, seed=20260928, device=\'cuda\',\n                selection_domain=\'real\', alpha=None, tversky_reduction=\'batch\', steps_per_epoch=None)\n\n\ndef setup(config):\n    c = {**DEFAULTS, **config}\n    if c[\'task\'] not in (\'crack\', \'corrosion\') or c[\'selection_domain\'] not in (\'real\', \'synthetic\'):\n        raise ValueError(\'Invalid task or selection_domain\')\n    if c[\'size\'] < 64 or c[\'size\'] % 32 or min(c[\'epochs\'], c[\'patience\'], c[\'batch_size\']) < 1:\n        raise ValueError(\'Invalid size/epoch/batch configuration\')\n    if not 0 < c[\'threshold\'] < 1 or not np.isfinite(c[\'lr\']) or c[\'lr\'] <= 0:\n        raise ValueError(\'Invalid threshold or learning rate\')\n    if not isinstance(c[\'normal_fp_pixels\'], int) or c[\'normal_fp_pixels\'] < 0:\n        raise ValueError(\'normal_fp_pixels must be a nonnegative integer\')\n    if c[\'alpha\'] is not None and not 0 < c[\'alpha\'] < 1:\n        raise ValueError(\'alpha must be between 0 and 1\')\n    if c[\'tversky_reduction\'] not in (\'batch\', \'image\'):\n        raise ValueError(\'Invalid Tversky reduction\')\n    if c[\'steps_per_epoch\'] is not None and (type(c[\'steps_per_epoch\']) is not int or c[\'steps_per_epoch\'] < 1):\n        raise ValueError(\'steps_per_epoch must be a positive integer or None\')\n    if c[\'device\'] not in (\'cuda\', \'cpu\'):\n        raise ValueError(\'device must be cuda or cpu\')\n    if c[\'device\'] == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'Colab GPU is not connected; no silent CPU fallback\')\n    random.seed(c[\'seed\']); np.random.seed(c[\'seed\']); torch.manual_seed(c[\'seed\'])\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(c[\'seed\'])\n    torch.set_num_threads(2)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.use_deterministic_algorithms(True)\n    return c\n\n\ndef new_model(weights, device):\n    model = smp.Unet(encoder_name=\'efficientnet-b0\', encoder_weights=None, in_channels=3, classes=1)\n    model.load_state_dict(checkpoint_state(weights), strict=True)\n    return model.to(device)\n\n\ndef comparison(baseline, tuned):\n    result = {}\n    for kind in (\'real\', \'synthetic\'):\n        b, t = baseline[\'groups\'][kind], tuned[\'groups\'][kind]\n        delta = {key: t[\'positive_macro\'][key]-b[\'positive_macro\'][key]\n                 if t[\'positive_macro\'][key] is not None and b[\'positive_macro\'][key] is not None else None\n                 for key in (\'iou\', \'dice\', \'precision\', \'recall\')}\n        result[kind] = dict(baseline=b, finetuned=t, delta=delta)\n    return result\n\n\ndef write_tables(out, task, split, baseline, tuned):\n    out = Path(out)\n    fields = [\'task\', \'split\', \'data_type\', \'model\', \'n_images\', \'source_groups\', \'positive_n\', \'normal_n\',\n              \'iou\', \'dice\', \'precision\', \'recall\', \'normal_false_positive_images\',\n              \'normal_false_positive_image_rate\', \'normal_mean_false_positive_area\']\n    tables = []\n    for name, result in [(\'baseline\', baseline), (\'finetuned\', tuned)]:\n        for kind, group in result[\'groups\'].items():\n            row = dict(task=task, split=split, data_type=kind, model=name,\n                       source_groups=len({r[\'source_group\'] for r in result[\'per_image\'] if r[\'data_type\']==kind}),\n                       **{k: group[k] for k in fields if k in group}, **group[\'positive_macro\'])\n            tables.append(row)\n    with (out/\'summary.csv\').open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as f:\n        writer = csv.DictWriter(f, fieldnames=fields); writer.writeheader(); writer.writerows(tables)\n    with (out/\'per_image.csv\').open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as f:\n        per = [dict(model=name, **row) for name, result in [(\'baseline\', baseline), (\'finetuned\', tuned)] for row in result[\'per_image\']]\n        writer = csv.DictWriter(f, fieldnames=list(per[0])); writer.writeheader(); writer.writerows(per)\n    def fmt(value):\n        return \'해당 없음\' if value is None else f\'{value:.4f}\' if isinstance(value, float) else str(value)\n    title = \'Validation 결과\' if split==\'val\' else \'최종 Test 결과\'\n    lines = [f\'# {task} {title}\', \'\', \'실제 사진과 합성 사진 분리 집계. IoU 등은 해당 결함이 있는 사진의 평균값임.\', \'\',\n             \'|자료|모델|사진 수|원본 그룹 수|IoU|Precision|Recall|정상 오탐 비율|정상 오탐 면적 비율|\',\n             \'|---|---|---:|---:|---:|---:|---:|---:|---:|\']\n    for row in tables:\n        lines.append(\'|\'+\'|\'.join(fmt(row[k]) for k in (\'data_type\',\'model\',\'n_images\',\'source_groups\',\'iou\',\'precision\',\'recall\',\'normal_false_positive_image_rate\',\'normal_mean_false_positive_area\'))+\'|\')\n    lines += [\'\', \'빈 평가 집단의 지표는 해당 없음으로 표시함. 위 비율의 범위는 0~1임.\',\n              \'단일 실행 결과이며 안전등급 또는 현장 성능 보장 근거가 아님.\',\n              \'Validation은 모델 선택에 사용한 자료이며 독립 Test 성능과 구분함.\' if split==\'val\' else \'이 결과 확인 후 같은 Test에 맞춘 모델 또는 기준 조정 금지.\']\n    (out/\'결과표.md\').write_text(\'\\n\'.join(lines)+\'\\n\', encoding=\'utf-8\')\n\n\ndef train(config, out):\n    c = setup(config)\n    root, out = Path(c[\'data\']).resolve(), Path(out).resolve()\n    if out.exists():\n        raise FileExistsError(\'Use a new run directory\')\n    all_rows = verify_prepared(root, {\'train\', \'val\'})\n    rows = [r for r in all_rows if c[\'task\'] in r[\'complete_targets\']]\n    splits = {s: [r for r in rows if r[\'split\']==s] for s in (\'train\',\'val\')}\n    for name, members in splits.items():\n        if not any(r[\'positive_pixels\'][c[\'task\']] for r in members) or not any(r[\'normal_confirmed\'] for r in members):\n            raise ValueError(f\'{name}: positive and confirmed normal images required for this task\')\n    selected = [r for r in splits[\'val\'] if r[\'data_type\']==c[\'selection_domain\']]\n    if not any(r[\'positive_pixels\'][c[\'task\']] for r in selected) or not any(r[\'normal_confirmed\'] for r in selected):\n        raise ValueError(\'selection_domain validation needs positive and normal images\')\n    weights = Path(c[\'weights\']).resolve()\n    model = new_model(weights, c[\'device\'])\n    train_ds = Samples(root, splits[\'train\'], c[\'task\'], c[\'size\'], True)\n    val_ds = Samples(root, splits[\'val\'], c[\'task\'], c[\'size\'])\n    encoder_before = {k:v.cpu().clone() for k,v in model.encoder.state_dict().items()}\n    for param in model.encoder.parameters(): param.requires_grad_(False)\n    out.mkdir(parents=True)\n    c.update(data=str(root), weights=str(weights), model=\'Unet/efficientnet-b0\',\n             encoder_frozen=True, encoder_batchnorm_frozen=True, mean=MEAN.tolist(), std=STD.tolist(),\n             dataset_sha256=sha(root/\'dataset.json\'), baseline_sha256=sha(weights),\n             torch=str(torch.__version__), smp=smp.__version__, python=platform.python_version(),\n             code_hashes={p.name:sha(p) for p in Path(__file__).parent.glob(\'*.py\')})\n    save_json(out/\'config.json\', c)\n    # Keep the complete train/val provenance, including classes not selected for this model.\n    save_json(out/\'trainval_provenance.json\', dict(items=all_rows))\n    ev = lambda dest=None: evaluate(model, val_ds, c[\'device\'], c[\'threshold\'], dest, c[\'normal_fp_pixels\'])\n    baseline = ev(out/\'baseline\')\n    save_json(out/\'baseline_val.json\', baseline)\n    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=c[\'lr\'], weight_decay=1e-4)\n    sampler = (RandomSampler(train_ds, replacement=True,\n                num_samples=c[\'steps_per_epoch\'] * c[\'batch_size\'],\n                generator=torch.Generator().manual_seed(c[\'seed\']))\n               if c[\'steps_per_epoch\'] is not None else None)\n    loader = DataLoader(train_ds, batch_size=c[\'batch_size\'], shuffle=sampler is None,\n                        sampler=sampler, num_workers=0)\n    best_score, best_epoch, stale, history = -1., 0, 0, []\n    start = time.monotonic()\n    for epoch in range(1, c[\'epochs\']+1):\n        model.train(); model.encoder.eval()\n        losses = []\n        for x,y,v in loader:\n            x,y,v = [a.to(c[\'device\']) for a in (x,y,v)]\n            optimizer.zero_grad(set_to_none=True)\n            loss = loss_value(model(x), y, v, c[\'task\'], c[\'alpha\'], c[\'tversky_reduction\'])\n            if not torch.isfinite(loss): raise FloatingPointError(\'Nonfinite loss\')\n            loss.backward()\n            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1., error_if_nonfinite=True)\n            optimizer.step(); losses.append(float(loss.detach()))\n        val = ev()\n        score = val[\'groups\'][c[\'selection_domain\']][\'positive_macro\'][\'iou\']\n        history.append(dict(epoch=epoch, optimizer_steps=len(losses), loss=float(np.mean(losses)), selection_iou=score, groups=val[\'groups\']))\n        save_json(out/\'history.json\', history)\n        print(f"{c[\'task\']} epoch={epoch} val_iou={score:.6f}", flush=True)\n        if score > best_score + 1e-8:\n            best_score, best_epoch, stale = score, epoch, 0\n            state = {k:v.detach().cpu() for k,v in model.state_dict().items()}\n            if not all(torch.isfinite(v).all() for v in state.values()): raise FloatingPointError(\'Nonfinite state\')\n            torch.save(state, out/\'best.tmp\'); (out/\'best.tmp\').replace(out/\'best.pt\')\n        else: stale += 1\n        if stale >= c[\'patience\']: break\n    for k,v in model.encoder.state_dict().items():\n        if not torch.equal(v.cpu(), encoder_before[k]): raise RuntimeError(\'Frozen encoder changed\')\n    if sha(weights) != c[\'baseline_sha256\']: raise RuntimeError(\'Baseline checkpoint changed\')\n    model.load_state_dict(checkpoint_state(out/\'best.pt\'), strict=True)\n    tuned = ev(out/\'finetuned\')\n    save_json(out/\'finetuned_val.json\', tuned)\n    save_json(out/\'result.json\', dict(status=\'completed\', split=\'val\', task=c[\'task\'], comparison=comparison(baseline,tuned),\n              best_epoch=best_epoch, epochs_run=len(history), elapsed_seconds=time.monotonic()-start,\n              best_sha256=sha(out/\'best.pt\'), config_sha256=sha(out/\'config.json\'),\n              provenance_sha256=sha(out/\'trainval_provenance.json\'), test_evaluated=False, deployment_replaced=False))\n    write_tables(out, c[\'task\'], \'val\', baseline, tuned)\n\n\ndef validate_run(run):\n    run = Path(run).resolve()\n    c = json.loads((run/\'config.json\').read_text())\n    r = json.loads((run/\'result.json\').read_text())\n    if r[\'status\'] != \'completed\': raise ValueError(\'Run has not completed\')\n    for name, expected in [(\'best.pt\',r[\'best_sha256\']),(\'config.json\',r[\'config_sha256\']),(\'trainval_provenance.json\',r[\'provenance_sha256\'])]:\n        if sha(run/name) != expected: raise ValueError(\'Run artifact changed: \'+name)\n    if sha(c[\'weights\']) != c[\'baseline_sha256\']: raise ValueError(\'Baseline checkpoint changed\')\n    for name, expected in c[\'code_hashes\'].items():\n        if sha(Path(__file__).parent/name) != expected: raise ValueError(\'Evaluation code changed since training: \'+name)\n    return c, r\n\n\ndef lock(run, note):\n    run = Path(run).resolve()\n    validate_run(run)\n    if not note.strip(): raise ValueError(\'Validation-based model-selection note required\')\n    path = run/\'evaluation_lock.json\'\n    payload = dict(result_sha256=sha(run/\'result.json\'), note=note, locked_at=time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()))\n    with path.open(\'x\', encoding=\'utf-8\') as f: json.dump(payload, f, ensure_ascii=False, indent=2)\n\n\ndef final_test(run, test_data, out, device=None):\n    run, root, out = Path(run).resolve(), Path(test_data).resolve(), Path(out).resolve()\n    c, result = validate_run(run)\n    locked = json.loads((run/\'evaluation_lock.json\').read_text())\n    if sha(run/\'result.json\') != locked[\'result_sha256\']: raise ValueError(\'Locked run changed\')\n    if out.exists(): raise FileExistsError(\'Use a new final evaluation directory\')\n    test_rows = verify_prepared(root, {\'test\'})\n    for row in test_rows:\n        if row.get(\'independence_reviewed\') is not True or row.get(\'used_for_training\') is not False or row.get(\'used_for_model_selection\') is not False:\n            raise ValueError(\'Test independence evidence missing\')\n    known = json.loads((run/\'trainval_provenance.json\').read_text())[\'items\']\n    leakage = check_leakage(known+test_rows)\n    if {r[\'id\'] for r in known} & {r[\'id\'] for r in test_rows}: leakage.append(\'Image IDs overlap\')\n    if leakage: raise ValueError(\'\\n\'.join(leakage))\n    task_rows = [r for r in test_rows if c[\'task\'] in r[\'complete_targets\']]\n    if not any(r[\'positive_pixels\'][c[\'task\']] for r in task_rows) or not any(r[\'normal_confirmed\'] for r in task_rows):\n        raise ValueError(\'Test requires target-positive and confirmed normal images\')\n    c = setup({**c, \'device\':device or c[\'device\']})\n    dataset = Samples(root, task_rows, c[\'task\'], c[\'size\'])\n    model = new_model(c[\'weights\'], c[\'device\'])\n    # Exclusive marker is deliberately created before predictions; failed runs stay recorded.\n    marker = run/\'test_evaluation_started.json\'\n    with marker.open(\'x\', encoding=\'utf-8\') as f:\n        json.dump(dict(test_manifest_sha256=sha(root/\'dataset.json\'), out=str(out), status=\'started\'), f, indent=2)\n    out.mkdir(parents=True)\n    baseline = evaluate(model, dataset, c[\'device\'], c[\'threshold\'], out/\'baseline\', c[\'normal_fp_pixels\'])\n    model.load_state_dict(checkpoint_state(run/\'best.pt\'), strict=True)\n    tuned = evaluate(model, dataset, c[\'device\'], c[\'threshold\'], out/\'finetuned\', c[\'normal_fp_pixels\'])\n    save_json(out/\'baseline_test.json\', baseline); save_json(out/\'finetuned_test.json\', tuned)\n    save_json(out/\'result.json\', dict(status=\'completed\', split=\'test\', task=c[\'task\'], comparison=comparison(baseline,tuned),\n              run_result_sha256=sha(run/\'result.json\'), evaluation_lock_sha256=sha(run/\'evaluation_lock.json\'),\n              test_manifest_sha256=sha(root/\'dataset.json\'), threshold=c[\'threshold\'], size=c[\'size\'],\n              normal_fp_pixels=c[\'normal_fp_pixels\'], deployment_replaced=False))\n    write_tables(out, c[\'task\'], \'test\', baseline, tuned)\n    save_json(run/\'test_evaluation_completed.json\', dict(result=str(out/\'result.json\'), result_sha256=sha(out/\'result.json\')))\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser(); sub = p.add_subparsers(dest=\'command\', required=True)\n    t = sub.add_parser(\'train\'); t.add_argument(\'--config\', type=Path, required=True); t.add_argument(\'--out\', type=Path, required=True)\n    l = sub.add_parser(\'lock\'); l.add_argument(\'--run\', type=Path, required=True); l.add_argument(\'--note\', required=True)\n    e = sub.add_parser(\'test\'); e.add_argument(\'--run\', type=Path, required=True); e.add_argument(\'--data\', type=Path, required=True); e.add_argument(\'--out\', type=Path, required=True); e.add_argument(\'--device\', choices=[\'cpu\',\'cuda\'])\n    a = p.parse_args()\n    if a.command == \'train\': train(json.loads(a.config.read_text()), a.out)\n    elif a.command == \'lock\': lock(a.run, a.note)\n    else: final_test(a.run, a.data, a.out, a.device)\n', 'experiments5.py': '"""Five conditions: two pretrained baselines and three independent fine-tuning runs.\nInput must have passed data.prepare; never auto-approve or reassign raw submissions.\n"""\nimport argparse\nimport json\nimport math\nimport shutil\nfrom collections import Counter\nfrom pathlib import Path\nfrom data import verify_prepared, save_json, sha\nfrom run import train, setup, new_model, final_test, lock\nfrom engine import Samples, evaluate\n\nRUNS = {\'crack_real\': (\'crack\', False), \'crack_real_synthetic\': (\'crack\', True),\n        \'corrosion_real\': (\'corrosion\', False)}\n\n\ndef select_conditions(rows):\n    result = {}\n    for name, (task, mixed) in RUNS.items():\n        selected = [r for r in rows if task in r[\'complete_targets\'] and\n                    ((r[\'split\'] == \'val\' and r[\'data_type\'] == \'real\') or\n                     (r[\'split\'] == \'train\' and (mixed or r[\'data_type\'] == \'real\')))]\n        for split in (\'train\', \'val\'):\n            part = [r for r in selected if r[\'split\'] == split]\n            if not any(r[\'positive_pixels\'][task] for r in part) or not any(r[\'normal_confirmed\'] for r in part):\n                raise ValueError(f\'{name}/{split}: target-positive and normal images required\')\n        result[name] = selected\n    if not any(r[\'split\'] == \'train\' and r[\'data_type\'] == \'synthetic\' and r[\'positive_pixels\'][\'crack\']\n               for r in result[\'crack_real_synthetic\']):\n        raise ValueError(\'Synthetic crack training data required for comparison\')\n    return result\n\n\ndef condition_data(source, destination, selected, all_rows):\n    destination.mkdir(parents=True, exist_ok=False)\n    for row in selected:\n        for name in row[\'file_hashes\']:\n            target = destination / name\n            if not target.resolve().is_relative_to(destination.resolve()):\n                raise ValueError(\'Unsafe prepared path\')\n            target.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(source / name, target)\n    save_json(destination/\'dataset.json\', dict(schema_version=1, kind=\'trainval\', items=selected))\n    save_json(destination/\'all_input_provenance.json\', dict(items=all_rows))\n\n\ndef execute(data, weights, out, epochs=20, seed=20260929, device=\'cuda\', size=512, batch_size=2):\n    root, out = Path(data).resolve(), Path(out).resolve()\n    if out.exists(): raise FileExistsError(\'Use a new experiment directory\')\n    rows = verify_prepared(root, {\'train\',\'val\'})\n    conditions = select_conditions(rows)\n    weights = {k: str(Path(v).resolve()) for k,v in weights.items()}\n    # Check both original checkpoints and all image/mask inputs before starting any run.\n    for task in (\'crack\',\'corrosion\'):\n        cfg = setup(dict(task=task, device=device, size=size, batch_size=batch_size, epochs=epochs, seed=seed))\n        model = new_model(weights[task], device)\n        del model\n    for name, selected in conditions.items():\n        Samples(root, selected, RUNS[name][0], size)\n    steps = max(1, math.ceil(sum(r[\'split\']==\'train\' for r in conditions[\'crack_real\']) / batch_size))\n    out.mkdir(parents=True)\n    plan = dict(status=\'started\', test_evaluated=False,\n                conditions=[\'crack_baseline\',\'crack_real\',\'crack_real_synthetic\',\'corrosion_baseline\',\'corrosion_real\'],\n                input_sha256=sha(root/\'dataset.json\'), weights={k:sha(v) for k,v in weights.items()},\n                seed=seed, rounds=epochs, crack_steps_per_round=steps,\n                crack_total_updates=steps*epochs,\n                note=\'Matched crack update budgets; sampling with replacement; best real Validation IoU selected. No Test used.\',\n                counts={name:dict(Counter(f"{r[\'split\']}/{r[\'data_type\']}" for r in rs)) for name,rs in conditions.items()})\n    save_json(out/\'experiment_plan.json\', plan)\n    save_json(out/\'all_trainval_provenance.json\', dict(items=rows))\n    for name, selected in conditions.items():\n        task, mixed = RUNS[name]\n        dest = out/\'condition_data\'/name\n        condition_data(root, dest, selected, rows)\n        config = dict(task=task, data=str(dest), weights=weights[task], device=device, size=size,\n                      batch_size=batch_size, epochs=epochs, seed=seed, selection_domain=\'real\')\n        if task == \'crack\':\n            # A round has the same update count in real-only and mixed conditions.\n            config.update(steps_per_epoch=steps, patience=epochs+1)\n        train(config, out/name)\n    result = {}\n    for name in RUNS:\n        result[name] = json.loads((out/name/\'finetuned_val.json\').read_text())\n    for task in (\'crack\',\'corrosion\'):\n        result[task+\'_baseline\'] = json.loads((out/(task+\'_real\')/\'baseline_val.json\').read_text())\n    # Both crack runs must have the same baseline predictions and Validation IDs.\n    other = json.loads((out/\'crack_real_synthetic\'/\'baseline_val.json\').read_text())\n    if result[\'crack_baseline\'] != other:\n        raise RuntimeError(\'Crack baseline evaluations differ across conditions\')\n    save_json(out/\'five_conditions_validation.json\', result)\n    plan[\'status\']=\'completed\'; save_json(out/\'experiment_plan.json\', plan)\n    return result\n\n\nif __name__ == \'__main__\':\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--data\',required=True,type=Path)\n    p.add_argument(\'--crack-weights\',required=True,type=Path)\n    p.add_argument(\'--corrosion-weights\',required=True,type=Path)\n    p.add_argument(\'--out\',required=True,type=Path)\n    p.add_argument(\'--epochs\',default=20,type=int)\n    p.add_argument(\'--seed\',default=20260929,type=int)\n    a=p.parse_args()\n    execute(a.data,dict(crack=a.crack_weights,corrosion=a.corrosion_weights),a.out,a.epochs,a.seed)\n', 'engine.py': '"""Binary U-Net core adapted from Subin\'s verified 2026-09-26 experiment.\nThe encoder, including its BatchNorm running state, is frozen by the runner.\n"""\nimport random\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image, ImageEnhance, ImageFilter\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader\nimport segmentation_models_pytorch as smp\nfrom metrics import metrics, grouped_summary\n\nMEAN = np.array([.485, .456, .406], dtype=np.float32)\nSTD = np.array([.229, .224, .225], dtype=np.float32)\n\n\ndef read_sample(root, row, task, size):\n    sid = row[\'id\']\n    with Image.open(root / row[\'image\']) as im:\n        rgb = im.convert(\'RGB\').resize((size, size), Image.Resampling.BILINEAR)\n    arrays = []\n    for name in (task, \'valid\'):\n        with Image.open(root / name / f\'{sid}.png\') as im:\n            arrays.append(np.array(im.resize((size, size), Image.Resampling.NEAREST)) > 0)\n    mask, valid = arrays\n    if not valid.any() or (row[\'positive_pixels\'][task] > 0 and not (mask & valid).any()):\n        raise ValueError(f\'{sid}: valid area or thin defect lost after resize; revise input size/patches\')\n    return np.asarray(rgb), mask, valid\n\n\ndef tensor(rgb):\n    return torch.from_numpy(np.ascontiguousarray(((rgb.astype(np.float32)/255 - MEAN)/STD).transpose(2, 0, 1)))\n\n\nclass Samples(Dataset):\n    def __init__(self, root, rows, task, size, augment=False):\n        self.rows = rows\n        self.samples = [read_sample(root, r, task, size) for r in rows]\n        self.augment = augment\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        rgb, mask, valid = [a.copy() for a in self.samples[index]]\n        if self.augment:\n            if random.random() < .5:\n                rgb, mask, valid = [np.flip(a, 1) for a in (rgb, mask, valid)]\n            if random.random() < .2:\n                rgb, mask, valid = [np.flip(a, 0) for a in (rgb, mask, valid)]\n            rotations = random.randrange(4)\n            rgb, mask, valid = [np.rot90(a, rotations) for a in (rgb, mask, valid)]\n            rgb = np.asarray(ImageEnhance.Brightness(Image.fromarray(rgb)).enhance(random.uniform(.9, 1.1)))\n            if random.random() < .2:\n                rgb = np.asarray(Image.fromarray(rgb).filter(ImageFilter.GaussianBlur(random.uniform(.2, .8))))\n        return (tensor(rgb), *[torch.from_numpy(np.ascontiguousarray(a[None], dtype=np.float32)) for a in (mask, valid)])\n\n\ndef loss_value(logits, y, valid, task, alpha=None, reduction=\'batch\'):\n    bce = (nn.functional.binary_cross_entropy_with_logits(logits, y, reduction=\'none\') * valid).sum()/valid.sum()\n    p = logits.sigmoid()\n    if reduction not in (\'batch\', \'image\'):\n        raise ValueError(\'Unknown Tversky reduction\')\n    axes = (0,1,2,3) if reduction == \'batch\' else (1,2,3)\n    tp = (p*y*valid).sum(axes)\n    fp = (p*(1-y)*valid).sum(axes)\n    fn = ((1-p)*y*valid).sum(axes)\n    alpha = (.3 if task == \'crack\' else .5) if alpha is None else alpha\n    beta = 1 - alpha\n    return .5*bce + .5*(1 - (tp+1)/(tp+alpha*fp+beta*fn+1)).mean()\n\n\ndef checkpoint_state(path):\n    obj = torch.load(path, map_location=\'cpu\', weights_only=True)\n    if isinstance(obj, dict):\n        for key in (\'state_dict\', \'model_state_dict\'):\n            if key in obj:\n                obj = obj[key]\n                break\n    if not isinstance(obj, dict) or not obj or not all(isinstance(v, torch.Tensor) for v in obj.values()):\n        raise ValueError(\'Expected tensor state_dict or state_dict/model_state_dict wrapper\')\n    if all(k.startswith(\'module.\') for k in obj):\n        obj = {k[7:]: v for k, v in obj.items()}\n    if not all(torch.isfinite(v).all() for v in obj.values()):\n        raise ValueError(\'Nonfinite checkpoint\')\n    return obj\n\n\n@torch.no_grad()\ndef evaluate(model, dataset, device, threshold, destination=None, normal_fp_pixels=0):\n    model.eval()\n    rows = []\n    if destination:\n        destination.mkdir(parents=True, exist_ok=True)\n    for i, record in enumerate(dataset.rows):\n        x, y, valid = dataset[i]\n        probability = model(x[None].to(device)).sigmoid()[0,0].cpu().numpy()\n        if not np.isfinite(probability).all():\n            raise FloatingPointError(\'Nonfinite prediction\')\n        pred = probability > threshold\n        gt, v = y[0].numpy() > 0, valid[0].numpy() > 0\n        rows.append(dict(id=record[\'id\'], source_group=record[\'source_group\'], data_type=record[\'data_type\'], normal_confirmed=record.get(\'normal_confirmed\', False), **metrics(pred, gt, v)))\n        if destination:\n            Image.fromarray(pred.astype(\'uint8\')*255).save(destination / f"{record[\'id\']}_prediction.png")\n            rgb = dataset.samples[i][0].copy()\n            overlay = rgb.copy()\n            overlay[pred & gt & v] = [0, 210, 0]\n            overlay[~pred & gt & v] = [255, 40, 40]\n            overlay[pred & ~gt & v] = [255, 210, 0]\n            overlay[~v] = [128, 128, 128]\n            panel = np.concatenate([rgb, np.repeat((gt*255).astype(\'uint8\')[...,None], 3, 2), overlay], axis=1)\n            Image.fromarray(panel).save(destination / f"{record[\'id\']}_comparison.jpg")\n    return dict(groups=grouped_summary(rows, normal_fp_pixels), per_image=rows)\n', 'import_metadata.py': '"""Adapt the team\'s agreed Notion metadata.csv to a versioned audit manifest.\n\nThis records a team\'s declared review, not an independent human verification.\nMissing/ambiguous review, prior-use or independence values stay unresolved.\n"""\nimport argparse\nimport csv\nfrom pathlib import Path\nfrom data import save_json, sha\n\n\ndef boolean(value):\n    value = str(value).strip().lower()\n    return {\'true\':True, \'false\':False, \'yes\':True, \'no\':False, \'1\':True, \'0\':False,\n            \'완료\':True, \'미완료\':False}.get(value)\n\n\ndef adapt(metadata, out, evidence):\n    metadata, out = Path(metadata).resolve(), Path(out).resolve()\n    if out.exists(): raise FileExistsError(out)\n    with metadata.open(encoding=\'utf-8-sig\', newline=\'\') as f: source = list(csv.DictReader(f))\n    rows = []\n    for raw in source:\n        row = dict(raw)\n        row[\'id\'] = raw.get(\'image_id\') or raw.get(\'id\')\n        for target, alias in [(\'image\',\'image_file\'), (\'labelme\',\'label_file\')]:\n            value = raw.get(alias) or raw.get(target)\n            row[target] = str((metadata.parent/value).resolve()) if value else \'\'\n        classes = set(raw.get(\'classes\',\'\').replace(\'+\',\'|\').replace(\',\',\'|\').split(\'|\'))\n        classes = {s.strip() for s in classes if s.strip()}\n        explicit_targets = raw.get(\'complete_targets\',\'\')\n        row[\'complete_targets\'] = explicit_targets.split(\'|\') if explicit_targets else ([\'crack\',\'corrosion\'] if classes=={\'normal\'} else sorted(classes & {\'crack\',\'corrosion\'}))\n        row[\'normal_confirmed\'] = classes=={\'normal\'}\n        row[\'image_reviewed\'] = boolean(raw.get(\'image_reviewed\', raw.get(\'reviewed\',\'\')))\n        row[\'annotation_reviewed\'] = boolean(raw.get(\'reviewed\',\'\'))\n        row[\'review_evidence\'] = evidence\n        row[\'review_source\'] = \'Team metadata declaration; not independently re-reviewed by this converter\'\n        row[\'source_metadata_sha256\'] = sha(metadata)\n        prior = raw.get(\'prior_use\',\'\').strip().lower()\n        known = {\'none\':(False,False), \'사용 없음\':(False,False), \'train\':(True,False),\n                 \'val\':(False,True), \'validation\':(False,True), \'model_selection\':(False,True),\n                 \'train+val\':(True,True)}\n        row[\'used_for_training\'], row[\'used_for_model_selection\'] = known.get(prior,(None,None))\n        row[\'independence_reviewed\'] = boolean(raw.get(\'independence_reviewed\',\'\'))\n        row[\'independence_evidence\'] = raw.get(\'independence_evidence\',\'\')\n        row[\'donor_groups\'] = [v.strip() for v in raw.get(\'defect_source_ids\',\'\').split(\'|\') if v.strip()]\n        row[\'background_groups\'] = [v.strip() for v in raw.get(\'background_source_id\',\'\').split(\'|\') if v.strip()]\n        for field in (\'train_only\', \'low_resolution_exception\'):\n            if raw.get(field, \'\').strip():\n                row[field] = boolean(raw[field])\n        if row[\'image_reviewed\'] is True and row[\'annotation_reviewed\'] is True:\n            for key in (\'image\',\'labelme\'):\n                if Path(row[key]).is_file(): row[\'reviewed_\'+key+\'_sha256\'] = sha(row[key])\n        rows.append(row)\n    save_json(out, dict(status=\'requires_data_audit\', items=rows))\n    return rows\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--metadata\', type=Path, required=True)\n    p.add_argument(\'--out\', type=Path, required=True)\n    p.add_argument(\'--review-evidence\', required=True, help=\'Team review record URL/version; conversion is not approval\')\n    a=p.parse_args()\n    print(\'Imported records:\',len(adapt(a.metadata,a.out,a.review_evidence)))\n', 'labelme_masks.py': '"""Polygon rasterization retained from the reviewed 2026-09-26 pipeline."""\nimport numpy as np\nfrom PIL import Image, ImageDraw\nTASKS = ("crack", "corrosion")\n\ndef rasterize(doc, size):\n    if (doc.get(\'imageWidth\'), doc.get(\'imageHeight\')) != size:\n        raise ValueError(\'LabelMe dimensions differ from image dimensions\')\n    layers = {k: Image.new(\'L\', size, 0) for k in (*TASKS, \'ignore\')}\n    for shape in doc[\'shapes\']:\n        label = shape[\'label\'].strip().lower()\n        # Preserve original graded JSON; merge only the binary training mask.\n        if label in (\'corrosion_fair\', \'corrosion_poor\', \'corrosion_severe\'):\n            label = \'corrosion\'\n        if label not in layers:\n            raise ValueError(f\'Unknown label: {label}; allowed: crack, corrosion, ignore\')\n        if shape.get(\'shape_type\') != \'polygon\':\n            raise ValueError(\'Only polygon shapes are supported; convert lines to reviewed polygons\')\n        pts = np.asarray(shape[\'points\'], dtype=float)\n        if pts.ndim != 2 or pts.shape[1] != 2 or len(pts) < 3 or not np.isfinite(pts).all():\n            raise ValueError(\'Invalid polygon points\')\n        if (pts < 0).any() or (pts[:, 0] >= size[0]).any() or (pts[:, 1] >= size[1]).any():\n            raise ValueError(\'Polygon outside image bounds\')\n        area = abs(np.dot(pts[:, 0], np.roll(pts[:, 1], 1)) - np.dot(pts[:, 1], np.roll(pts[:, 0], 1))) / 2\n        if area <= 0:\n            raise ValueError(\'Zero-area polygon\')\n        ImageDraw.Draw(layers[label]).polygon([tuple(p) for p in pts], fill=1)\n    valid = np.array(layers[\'ignore\']) == 0\n    if not valid.any():\n        raise ValueError(\'Image is entirely ignored\')\n    masks = {k: (np.array(layers[k]) > 0) & valid for k in TASKS}\n    return masks, valid\n', 'data.py': '"""Merge reviewed team manifests without silently splitting or approving samples."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport re\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom labelme_masks import TASKS, rasterize\n\nBOOL_FIELDS = (\'image_reviewed\', \'annotation_reviewed\', \'normal_confirmed\',\n               \'used_for_training\', \'used_for_model_selection\', \'independence_reviewed\', \'train_only\',\n               \'low_resolution_exception\')\nLIST_FIELDS = (\'complete_targets\', \'donor_groups\', \'background_groups\')\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n    temp.replace(path)\n\n\ndef read_manifests(paths):\n    rows = []\n    for path in paths:\n        path = Path(path).resolve()\n        if path.suffix.lower() == \'.csv\':\n            with path.open(encoding=\'utf-8-sig\', newline=\'\') as f:\n                items = list(csv.DictReader(f))\n            for row in items:\n                for key in BOOL_FIELDS:\n                    if row.get(key, \'\').strip().lower() in (\'true\', \'false\'):\n                        row[key] = row[key].strip().lower() == \'true\'\n                for key in LIST_FIELDS:\n                    row[key] = [s.strip() for s in row.get(key, \'\').split(\'|\') if s.strip()]\n        else:\n            obj = json.loads(path.read_text(encoding=\'utf-8\'))\n            items = obj[\'items\'] if isinstance(obj, dict) else obj\n        for item in items:\n            row = dict(item, manifest=str(path))\n            for key in (\'image\', \'labelme\'):\n                row[key] = str((path.parent / row[key]).resolve()) if row.get(key) else \'\'\n            rows.append(row)\n    return rows\n\n\ndef lineage(row):\n    """Global keys, never participant-prefixed: donors can also appear as real images."""\n    keys = set()\n    for key in (\'source_group\', \'original_group\', \'source_url\', \'source_image_url\'):\n        if row.get(key):\n            keys.add(\'source:\' + str(row[key]).strip())\n    for key in (\'donor_groups\', \'background_groups\'):\n        for value in row.get(key, []):\n            keys.add(\'source:\' + value.strip())\n    for key in (\'original_sha256\', \'image_sha256\', \'pixel_sha256\'):\n        if row.get(key):\n            keys.add(\'hash:\' + row[key])\n    return keys\n\n\ndef check_leakage(rows):\n    issues = []\n    groups = defaultdict(list)\n    for row in rows:\n        for token in lineage(row):\n            groups[token].append(row)\n    for token, members in groups.items():\n        splits = {r.get(\'split\') for r in members}\n        if len(splits) > 1:\n            issues.append(f"Split leakage {token}: " + \', \'.join(r[\'id\'] for r in members))\n        if \'test\' in splits and any(r.get(\'used_for_training\') is True or\n                                    r.get(\'used_for_model_selection\') is True for r in members):\n            issues.append(f"Test lineage has prior training/model-selection exposure: {token}")\n    return issues\n\n\ndef audit(rows, min_side=512, allow_reviewed_low_resolution=False):\n    errors, warnings, checked = [], [], []\n    seen_ids, seen_pixels = set(), {}\n    for item in rows:\n        row = dict(item)\n        sid = row.get(\'id\', \'?\')\n        try:\n            if not isinstance(sid, str) or not re.fullmatch(r\'(SB|ES|HM)_[A-Za-z0-9_-]+\', sid) or sid in seen_ids:\n                raise ValueError(\'unique ID with SB_, ES_ or HM_ prefix required\')\n            seen_ids.add(sid)\n            if row.get(\'split\') not in (\'train\', \'val\', \'test\'):\n                raise ValueError(\'explicit split train, val or test required; no automatic reassignment\')\n            if row.get(\'data_type\') not in (\'real\', \'synthetic\'):\n                raise ValueError(\'data_type must be real or synthetic\')\n            for key in (\'source_group\', \'source_url\', \'review_evidence\'):\n                if not isinstance(row.get(key), str) or not row[key].strip():\n                    raise ValueError(f\'{key} required\')\n            for key in (\'image_reviewed\', \'annotation_reviewed\'):\n                if row.get(key) is not True:\n                    raise ValueError(f\'{key} must be true; drafts are not approved\')\n            for key in (\'used_for_training\', \'used_for_model_selection\'):\n                if not isinstance(row.get(key), bool):\n                    raise ValueError(f\'{key} must explicitly be true or false\')\n            if row.get(\'train_only\') and row[\'split\'] != \'train\':\n                raise ValueError(\'train_only sample cannot enter val/test\')\n            if row[\'split\'] == \'test\':\n                if row[\'used_for_training\'] or row[\'used_for_model_selection\']:\n                    raise ValueError(\'Test sample was previously used for training or model selection\')\n                if row.get(\'independence_reviewed\') is not True or not row.get(\'independence_evidence\'):\n                    raise ValueError(\'Test independence review and evidence required, including donor/background history\')\n            for key in LIST_FIELDS:\n                if not isinstance(row.get(key, []), list) or any(not isinstance(v, str) or not v.strip() for v in row.get(key, [])):\n                    raise ValueError(f\'{key} must be a list of nonempty strings\')\n            if row[\'data_type\'] == \'synthetic\':\n                if not row.get(\'background_groups\'):\n                    raise ValueError(\'synthetic background_groups required\')\n                if row.get(\'normal_confirmed\') is not True and not row.get(\'donor_groups\'):\n                    raise ValueError(\'synthetic defect donor_groups required\')\n            for key in (\'image\', \'labelme\'):\n                row[key + \'_sha256\'] = sha(row[key])\n                if row.get(\'reviewed_\' + key + \'_sha256\') != row[key + \'_sha256\']:\n                    raise ValueError(f\'{key} review hash missing or file changed since approval\')\n            with Image.open(row[\'image\']) as im:\n                rgb = im.convert(\'RGB\')\n                size = rgb.size\n                if im.getexif().get(274, 1) != 1:\n                    raise ValueError(\'EXIF orientation must be normalized before annotation\')\n                row[\'pixel_sha256\'] = hashlib.sha256(str(size).encode() + rgb.tobytes()).hexdigest()\n            if min(size) < min_side:\n                if (allow_reviewed_low_resolution and row.get(\'low_resolution_exception\') is True\n                        and isinstance(row.get(\'low_resolution_reason\'), str)\n                        and row[\'low_resolution_reason\'].strip()):\n                    warnings.append(f\'{sid}: reviewed low-resolution exception {size}: {row["low_resolution_reason"]}\')\n                else:\n                    raise ValueError(f\'original image {size} below minimum side {min_side}; explicit reviewed exception required, no upscaling workaround\')\n            if row.get(\'upsampled\') is True:\n                raise ValueError(\'upsampled original is not eligible\')\n            previous = seen_pixels.get(row[\'pixel_sha256\'])\n            if previous:\n                raise ValueError(f\'duplicate decoded image of {previous}\')\n            seen_pixels[row[\'pixel_sha256\']] = sid\n            doc = json.loads(Path(row[\'labelme\']).read_text(encoding=\'utf-8\'))\n            if Path(doc.get(\'imagePath\', \'\')).name != Path(row[\'image\']).name:\n                raise ValueError(\'LabelMe imagePath does not match image filename\')\n            masks, valid = rasterize(doc, size)\n            complete = row.get(\'complete_targets\', [])\n            if not complete or not set(complete).issubset(TASKS):\n                raise ValueError(\'complete_targets must explicitly name fully reviewed classes\')\n            if row.get(\'valid_region_policy\') == \'reviewed_positive_only\':\n                if row[\'split\'] != \'train\' or row.get(\'train_only\') is not True:\n                    raise ValueError(\'partial positive-only annotation is train-only\')\n                valid &= np.logical_or.reduce(list(masks.values()))\n            elif row.get(\'valid_region_policy\', \'full\') != \'full\':\n                raise ValueError(\'unknown valid_region_policy\')\n            if not valid.any():\n                raise ValueError(\'no valid pixels\')\n            if row.get(\'normal_confirmed\') is True:\n                if doc[\'shapes\'] or set(complete) != set(TASKS):\n                    raise ValueError(\'normal requires empty shapes and both classes reviewed\')\n            elif not any(masks[t].any() for t in complete):\n                raise ValueError(\'empty annotations are not negatives without normal confirmation\')\n            if not isinstance(row.get(\'normal_confirmed\'), bool):\n                raise ValueError(\'normal_confirmed must be true or false\')\n            row[\'positive_pixels\'] = {t: int((m & valid).sum()) for t, m in masks.items()}\n            row[\'valid_pixels\'] = int(valid.sum())\n            row[\'width\'], row[\'height\'] = size\n            checked.append(row)\n        except (ValueError, KeyError, TypeError, OSError) as exc:\n            errors.append(f\'{sid}: {exc}\')\n    if not rows:\n        errors.append(\'No samples supplied\')\n    # Include invalid rows in metadata leakage checks, too, when keys are well formed.\n    safe_rows = [r for r in rows if all(isinstance(r.get(k, []), list) for k in (\'donor_groups\', \'background_groups\'))]\n    try:\n        errors.extend(check_leakage(safe_rows + checked))\n    except (TypeError, AttributeError):\n        errors.append(\'Malformed provenance identifiers\')\n    warnings.append(\'Hashes detect exact duplicates only. Human source/crop/scene checks remain required.\')\n    report = dict(passed=not errors, n_input=len(rows), n_valid=len(checked), errors=sorted(set(errors)),\n                  warnings=warnings, counts=dict(Counter(f"{r[\'split\']}/{r[\'data_type\']}" for r in checked)))\n    return checked, report\n\n\ndef verify_prepared(root, allowed_splits):\n    root = Path(root)\n    doc = json.loads((root / \'dataset.json\').read_text(encoding=\'utf-8\'))\n    rows = doc[\'items\']\n    if not rows or any(r[\'split\'] not in allowed_splits for r in rows):\n        raise ValueError(\'Unexpected split: train/val and Test must be physically separated\')\n    errors = check_leakage(rows)\n    for row in rows:\n        for name, expected in row[\'file_hashes\'].items():\n            path = (root / name).resolve()\n            if not path.is_relative_to(root.resolve()) or sha(path) != expected:\n                errors.append(f"Prepared data changed: {row[\'id\']}/{name}")\n    if errors:\n        raise ValueError(\'\\n\'.join(errors))\n    return rows\n\n\ndef prepare(rows, out, min_side=512, allow_reviewed_low_resolution=False):\n    checked, report = audit(rows, min_side, allow_reviewed_low_resolution)\n    if not report[\'passed\']:\n        raise ValueError(\'\\n\'.join(report[\'errors\']))\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=False)\n    save_json(out / \'audit.json\', report)\n    for kind, splits in [(\'trainval\', {\'train\', \'val\'}), (\'test_sealed\', {\'test\'})]:\n        selected = [r for r in checked if r[\'split\'] in splits]\n        if not selected:\n            continue\n        root = out / kind\n        for folder in (\'images\', \'labelme\', \'crack\', \'corrosion\', \'valid\'):\n            (root / folder).mkdir(parents=True)\n        exported = []\n        for row in selected:\n            sid = row[\'id\']\n            with Image.open(row[\'image\']) as im:\n                rgb = im.convert(\'RGB\')\n            label = json.loads(Path(row[\'labelme\']).read_text(encoding=\'utf-8\'))\n            masks, valid = rasterize(label, rgb.size)\n            if row.get(\'valid_region_policy\') == \'reviewed_positive_only\':\n                valid &= np.logical_or.reduce(list(masks.values()))\n            rgb.save(root / \'images\' / f\'{sid}.png\')\n            for name, mask in {**masks, \'valid\': valid}.items():\n                Image.fromarray(mask.astype(\'uint8\') * 255).save(root / name / f\'{sid}.png\')\n            label[\'imagePath\'] = f\'../images/{sid}.png\'\n            label[\'imageData\'] = None\n            save_json(root / \'labelme\' / f\'{sid}.json\', label)\n            item = {**row, \'image\': f\'images/{sid}.png\', \'labelme\': f\'labelme/{sid}.json\'}\n            item[\'file_hashes\'] = {f\'{k}/{sid}.{ext}\': sha(root / k / f\'{sid}.{ext}\')\n                                  for k, ext in [(\'images\', \'png\'), (\'labelme\', \'json\'), (\'crack\', \'png\'), (\'corrosion\', \'png\'), (\'valid\', \'png\')]}\n            exported.append(item)\n        save_json(root / \'dataset.json\', dict(schema_version=1, kind=kind, items=exported))\n    return report\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'command\', choices=[\'audit\', \'prepare\'])\n    p.add_argument(\'--manifests\', nargs=\'+\', type=Path, required=True)\n    p.add_argument(\'--out\', type=Path, required=True)\n    p.add_argument(\'--allow-reviewed-low-resolution\', action=\'store_true\',\n                   help=\'Accept only reviewed rows with low_resolution_exception=true and a recorded reason\')\n    a = p.parse_args()\n    rows = read_manifests(a.manifests)\n    if a.command == \'audit\':\n        _, report = audit(rows, allow_reviewed_low_resolution=a.allow_reviewed_low_resolution)\n        save_json(a.out, report)\n        print(json.dumps(report, ensure_ascii=False, indent=2))\n        raise SystemExit(0 if report[\'passed\'] else 2)\n    print(json.dumps(prepare(rows, a.out, allow_reviewed_low_resolution=a.allow_reviewed_low_resolution), ensure_ascii=False))\n', 'requirements.txt': 'segmentation-models-pytorch==0.5.0\nnumpy>=1.26,<3\nPillow>=10,<13\n# Use the matched PyTorch/torchvision pair already supplied by the Colab GPU runtime.\n', 'templates/crack_config.json': '{\n  "task": "crack",\n  "data": "/content/drive/MyDrive/ICT_통합실험0928/prepared_trainval_v1/trainval",\n  "weights": "/content/drive/MyDrive/ICT_통합실험0928/weights/best_E5_tversky.pt",\n  "size": 512,\n  "epochs": 20,\n  "patience": 5,\n  "batch_size": 2,\n  "lr": 3e-05,\n  "threshold": 0.5,\n  "normal_fp_pixels": 0,\n  "seed": 20260928,\n  "device": "cuda",\n  "selection_domain": "real",\n  "alpha": 0.3,\n  "tversky_reduction": "batch"\n}\n', 'templates/corrosion_config.json': '{\n  "task": "corrosion",\n  "data": "/content/drive/MyDrive/ICT_통합실험0928/prepared_trainval_v1/trainval",\n  "weights": "/content/drive/MyDrive/ICT_통합실험0928/weights/best_corrosion_efficientnet-b0.pt",\n  "size": 512,\n  "epochs": 20,\n  "patience": 5,\n  "batch_size": 2,\n  "lr": 3e-05,\n  "threshold": 0.5,\n  "normal_fp_pixels": 0,\n  "seed": 20260928,\n  "device": "cuda",\n  "selection_domain": "real",\n  "alpha": 0.5,\n  "tversky_reduction": "batch"\n}\n'}
for name, content in CODE_FILES.items():
    p = CODE / name
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding='utf-8')
os.chdir(CODE)
if str(CODE) not in sys.path: sys.path.insert(0, str(CODE))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE/'requirements.txt')], check=True)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '미연결: 학습 전 T4 선택 필요')


## 입력 설정
각 ZIP은 images, labelme, metadata.csv, README.md를 포함하는 검수 완료 제출본이어야 함. Test 파일은 아래 학습 목록에 넣지 않음. 검수 기록 링크는 실제 전달 기록으로 변경 필요함.

In [ ]:
RUN_TRAINING = False  # 팀 자료 및 검수 기록 확정 후에만 True
TRAINVAL_ZIPS = {
    'SB': BASE / 'inputs/subin_trainval_20260929_v4.zip',
    'ES': BASE / 'inputs/eunsu_trainval_20260929_v5.zip',
    'HM': BASE / 'inputs/hyemin_trainval_20260929_v5.zip',
}
REVIEW_RECORDS = {'SB':'미입력', 'ES':'미입력', 'HM':'미입력'}
WEIGHTS = {
    'crack': BASE / 'weights/best_E5_tversky.pt',
    'corrosion': BASE / 'weights/best_corrosion_efficientnet-b0.pt',
}
RUN_VERSION = 'team_20260929_v1'
ALLOW_REVIEWED_LOW_RESOLUTION = False  # 승인 기록이 있는 작은 사진을 사용할 때만 True

# ES/HM 파일명은 최종 수신본으로 교체 필요함. 낮은 해상도 허용은 검수 기록 확인 후 변경함.
for task, path in WEIGHTS.items():
    print(task, "가중치 있음" if path.is_file() else "가중치 업로드 필요", path)


In [ ]:
from import_metadata import adapt
from data import read_manifests, audit, prepare, save_json

def unpack_metadata(archive, destination):
    destination.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Invalid ZIP path')
        z.extractall(destination)
    files = [p for p in destination.rglob('metadata.csv') if '__MACOSX' not in p.parts]
    if len(files) != 1:
        raise ValueError('ZIP마다 metadata.csv 1개 필요함')
    return files[0]

def import_packages(packages, kind):
    for owner in packages:
        if REVIEW_RECORDS.get(owner, '미입력') == '미입력':
            raise ValueError(owner + ': 실제 검수 기록 링크 입력 필요함')
    manifests = []
    for owner, archive in packages.items():
        dest = BASE / f'unpacked_{RUN_VERSION}' / kind / owner
        metadata = unpack_metadata(archive, dest)
        manifest = metadata.parent / 'manifest_v1.json'
        adapt(metadata, manifest, REVIEW_RECORDS[owner])
        manifests.append(manifest)
    return manifests

if not RUN_TRAINING:
    print('팀 데이터 미확정: 데이터 준비 및 학습 대기')
else:
    train_manifests = import_packages(TRAINVAL_ZIPS, 'trainval')
    train_rows = read_manifests(train_manifests)
    if any(r['split'] not in ('train', 'val') for r in train_rows):
        raise ValueError('학습 ZIP에 Test 또는 미확정 분할이 포함됨')
    checked, report = audit(train_rows, allow_reviewed_low_resolution=ALLOW_REVIEWED_LOW_RESOLUTION)
    save_json(BASE / f'audit_trainval_{RUN_VERSION}.json', report)
    print(json.dumps(report, ensure_ascii=False, indent=2))
    if not report['passed']:
        raise ValueError('오류 수정 후 새 버전으로 준비 필요함')
    prepared = BASE / f'prepared_trainval_{RUN_VERSION}'
    prepare(train_rows, prepared, allow_reviewed_low_resolution=ALLOW_REVIEWED_LOW_RESOLUTION)


## 기본 모델2개 및 파인튜닝3개
균열 실제만/실제+합성, 부식 실제만 학습함. 균열 두 조건은 같은 실제 Validation과 동일 업데이트 예산 사용함.


In [ ]:
from experiments5 import execute, RUNS
if not RUN_TRAINING:
    print('학습 대기: 검수 및 분할 확정 필요')
else:
    experiment_dir = BASE / 'runs' / RUN_VERSION
    five_results = execute(prepared / 'trainval', WEIGHTS, experiment_dir)
    runs = {name: experiment_dir/name for name in RUNS}
    for name, run_path in runs.items():
        print(name, (run_path/'결과표.md').read_text())


## 최종 Test
위 Validation 결과를 검토하고 모델 및 평가 기준을 확정한 뒤 실행 필요함. 마지막 셀은 기본적으로 비활성화 상태임. Test 결과로 재학습하거나 임계값을 조정하지 않음. 실제 Test와 합성 Test 결과는 별도 산출함.

In [ ]:
RUN_FINAL_TEST = False
VALIDATION_DECISION = ''  # 최종 평가 대상 모델로 선택한 근거 기록
TEST_ZIPS = {
    'SB': BASE / 'inputs/subin_test_20260929_v5.zip',
    'ES': BASE / 'inputs/eunsu_test_20260929_v2.zip',
    'HM': BASE / 'inputs/hyemin_test_20260929_v3.zip',
}


In [ ]:
from run import lock, final_test
if not RUN_FINAL_TEST:
    print('최종 Test 미실행. Validation 검토와 자료 확정 후에만 활성화 필요함.')
else:
    if not VALIDATION_DECISION.strip():
        raise ValueError('Validation 검토 근거 입력 필요함')
    test_manifests = import_packages(TEST_ZIPS, 'test')
    test_rows = read_manifests(test_manifests)
    if any(r['split'] != 'test' for r in test_rows):
        raise ValueError('Test ZIP 분할 오류')
    _, report = audit(train_rows + test_rows, allow_reviewed_low_resolution=ALLOW_REVIEWED_LOW_RESOLUTION)
    save_json(BASE / f'audit_all_{RUN_VERSION}.json', report)
    if not report['passed']:
        raise ValueError(json.dumps(report, ensure_ascii=False, indent=2))
    test_prepared = BASE / f'prepared_test_{RUN_VERSION}'
    prepare(test_rows, test_prepared, allow_reviewed_low_resolution=ALLOW_REVIEWED_LOW_RESOLUTION)
    for task in ('crack_real','crack_real_synthetic','corrosion_real'):
        lock(runs[task], VALIDATION_DECISION)
        out = BASE / 'results' / RUN_VERSION / (task + '_test')
        final_test(runs[task], test_prepared / 'test_sealed', out)
        print((out / '결과표.md').read_text())


Drive의 runs와 results에 실행 설정, 최고 가중치, 점수 JSON, CSV, 결과표 및 비교 그림 저장됨. 기존 실행 폴더를 덮어쓰지 않으며 재실행은 새 RUN_VERSION으로 구분 필요함. Test를 이미 실행한 후에는 버전만 바꾸어 같은 Test에 맞춘 재조정 금지임.